In [229]:
import pandas as pd
import numpy as np


In [230]:
df = pd.read_csv(r"C:\Users\ADYA TRIPATHI\Downloads\ML datsaets\Bengaluru_House_Data.csv")
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


Understanding the data,cleaning and filling missing values

In [231]:
df.shape

(13320, 9)

In [232]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [233]:
#value count on each column
for column in df.columns:
    print(df[column].value_counts())
    print('\n')    

area_type
Super built-up  Area    8790
Built-up  Area          2418
Plot  Area              2025
Carpet  Area              87
Name: count, dtype: int64


availability
Ready To Move    10581
18-Dec             307
18-May             295
18-Apr             271
18-Aug             200
                 ...  
15-Aug               1
17-Jan               1
16-Nov               1
16-Jan               1
14-Jul               1
Name: count, Length: 81, dtype: int64


location
Whitefield                        540
Sarjapur  Road                    399
Electronic City                   302
Kanakpura Road                    273
Thanisandra                       234
                                 ... 
Bapuji Layout                       1
1st Stage Radha Krishna Layout      1
BEML Layout 5th stage               1
singapura paradise                  1
Abshot Layout                       1
Name: count, Length: 1305, dtype: int64


size
2 BHK         5199
3 BHK         4310
4 Bedroom      826
4 BHK    

In [234]:
df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [235]:
# area_type : no missing values 4 categories
# availability : no missing value many categories
# location : 1 missing , many categories
# size : 16 missing values and bedroom-bhk problem
# society : a lot of missing values 
# total_sqft : no missing but some values are in range we have to replace them with mean of range
# bath : 73 missing
# balcony : 609 missing 4 categories
# price : no missing

In [236]:
df.drop(columns = ['area_type','availability','society','balcony'],inplace = True)

In [237]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  object 
 1   size        13304 non-null  object 
 2   total_sqft  13320 non-null  object 
 3   bath        13247 non-null  float64
 4   price       13320 non-null  float64
dtypes: float64(2), object(3)
memory usage: 520.4+ KB


In [238]:
# now we will fill the missing values
# only one missing fill with any random value
df['location'] = df['location'].fillna('Sarjapur  Road')

In [239]:
df['size'] = df['size'].fillna('2 BHK')

In [240]:
df['bath'] = df['bath'].fillna(df['bath'].median())

In [241]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13320 non-null  object 
 1   size        13320 non-null  object 
 2   total_sqft  13320 non-null  object 
 3   bath        13320 non-null  float64
 4   price       13320 non-null  float64
dtypes: float64(2), object(3)
memory usage: 520.4+ KB


Feature Engineering
1 - Creating new column bhk 
2 - total_sqft : remove ranges
3 - fix location column bcuz its categorical will have to ohe it has many categories
4 - outlier detection and removal(sqft/bhk)

In [242]:
df['bhk'] = df['size'].str.split().str.get(0).astype(int)
df.drop(columns = ['size'],inplace = True) # Don't need size now

In [243]:
# only 2 such rows...outliers of the data
df[df['bhk'] > 20]

,location,total_sqft,bath,price,bhk
1718,2Electronic City Phase II,8000,27.0,230.0,27
4684,Munnekollal,2400,40.0,660.0,43


In [244]:
df.head()

,location,total_sqft,bath,price,bhk
0,Electronic City Phase II,1056,2.0,39.07,2
1,Chikka Tirupathi,2600,5.0,120.00,4
2,Uttarahalli,1440,2.0,62.00,3
3,Lingadheeranahalli,1521,3.0,95.00,3
4,Kothanur,1200,2.0,51.00,2


In [245]:
def convert_range(x):
    temp = x.split('-')
    if(len(temp) == 2):
        return (float(temp[0]) + float(temp[1]))/2
    try:
        return float(x)
    except:
        return None
    
df['total_sqft'] = df['total_sqft'].apply(convert_range)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13320 non-null  object 
 1   total_sqft  13274 non-null  float64
 2   bath        13320 non-null  float64
 3   price       13320 non-null  float64
 4   bhk         13320 non-null  int64  
dtypes: float64(3), int64(1), object(1)
memory usage: 520.4+ KB


In [246]:
df['price_per_sqft'] = df['total_sqft']*100000/df['price']
df.head()

,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,2.702841e+06
1,Chikka Tirupathi,2600.0,5.0,120.00,4,2.166667e+06
2,Uttarahalli,1440.0,2.0,62.00,3,2.322581e+06
3,Lingadheeranahalli,1521.0,3.0,95.00,3,1.601053e+06
4,Kothanur,1200.0,2.0,51.00,2,2.352941e+06


In [247]:
location_count = df['location'].value_counts()
location_count


location
Whitefield                        540
Sarjapur  Road                    400
Electronic City                   302
Kanakpura Road                    273
Thanisandra                       234
                                 ... 
Bapuji Layout                       1
1st Stage Radha Krishna Layout      1
BEML Layout 5th stage               1
singapura paradise                  1
Abshot Layout                       1
Name: count, Length: 1305, dtype: int64

In [248]:
location_count_less_than10 = location_count[location_count <= 10]
df['location'] = df['location'].apply(lambda x : 'other' if x in location_count_less_than10 else x)
df['location'].value_counts()

location
other                 2900
Whitefield             540
Sarjapur  Road         400
Electronic City        302
Kanakpura Road         273
                      ... 
Marsur                  11
Banjara Layout          11
LB Shastri Nagar        11
Pattandur Agrahara      11
Narayanapura            11
Name: count, Length: 242, dtype: int64

In [249]:
df.shape

(13320, 6)

In [256]:
df['sqft_per_bhk'] = df['total_sqft']/df['bhk']
df['sqft_per_bhk'].describe()
#we see min as 0.25 -> there is sqft / bhk is 0.25 which is definety an outlier
#minimum outlier -> les than 300(highly unlivable commercial/slum space mislabeled as a standard flat.)
#maximum outlier -> more than 2000(Luxury / Errors)

count    13274.000000
mean       575.074878
std        388.205175
min          0.250000
25%        473.333333
50%        552.500000
75%        625.000000
max      26136.000000
Name: sqft_per_bhk, dtype: float64

In [ ]:
# Force-overwriting df explicitly to ensure the shape updates
df = df[(df['sqft_per_bhk'] >= 300) & (df['sqft_per_bhk'] <= 2000)]
df.describe()

(12493, 7)